# CXR-LLaVA Colab T4 smoke test

Research-only notebook for loading CXR-LLaVA v2 and generating one report from a sample chest X-ray. Enable a Google Colab GPU runtime and select a T4 when available.

> Model output is experimental and must not be used for clinical diagnosis, treatment, or patient-specific medical decisions.

## 1. Configure the fork

Set `GITHUB_REPO_URL` to your public fork. Leave it empty only when the notebook is opened from an already cloned checkout. Do not put access tokens in this notebook.

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/hanhpm/CXR_LLaVA_Improvement.git"
GITHUB_BRANCH = "master"
PROJECT_DIR = Path("/content/CXR_LLaVA_Improvement")
MODEL_ID = "ECOFRI/CXR-LLAVA-v2"
SAMPLE_IMAGE = PROJECT_DIR / "IMG" / "img.jpg"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print("Using existing checkout:", PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

In [ ]:
%pip uninstall -y transformers tokenizers
%pip install -q \
    "transformers==4.46.3" \
    "tokenizers<0.21" \
    "huggingface-hub==0.36.0" \
    "protobuf==5.29.6" \
    sentencepiece \
    accelerate \
    pillow

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("Warning: no CUDA GPU detected; model loading may exceed CPU memory and is not recommended.")

In [ ]:
from IPython.display import display
from PIL import Image

if not SAMPLE_IMAGE.exists():
    raise FileNotFoundError(f"Sample image not found: {SAMPLE_IMAGE}")
sample_image = Image.open(SAMPLE_IMAGE)
print("Image:", SAMPLE_IMAGE)
print("Original mode and size:", sample_image.mode, sample_image.size)
display(sample_image.convert("L"))

## 2. Load the model

The checkpoint is large. The T4 path uses float16 and low CPU-memory loading to reduce peak memory. The first run downloads the model from Hugging Face and can take several minutes.

In [ ]:
import torch
import transformers
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
from unittest.mock import patch

original_from_pretrained = transformers.LlamaTokenizer.from_pretrained

def patched_from_pretrained(*args, **kwargs):
    kwargs.pop("add_special_tokens", None)
    return original_from_pretrained(*args, **kwargs)

with patch.object(
    transformers.LlamaTokenizer,
    "from_pretrained",
    new=patched_from_pretrained,
):
    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        device_map="auto" if device == "cuda" else None,
    )
model.eval()
print("Loaded:", MODEL_ID)
print("Device map: auto" if device == "cuda" else "Device: cpu")

In [ ]:
# Single-image smoke test using the documented repository API.
with torch.inference_mode():
    report = model.write_radiologic_report(sample_image)

print("MODEL-GENERATED REPORT (research only):")
print(report)

In [ ]:
# Optional question-answering smoke test.
question = "What findings should be reviewed by a qualified radiologist?"
with torch.inference_mode():
    answer = model.ask_question(question=question, image=sample_image)
print("QUESTION:", question)
print("MODEL-GENERATED ANSWER (research only):")
print(answer)

## Result classification

If the image loads and the two calls return text, this notebook has passed a **single-image inference smoke test**. It is not a benchmark, clinical validation, or metric-complete evaluation. The CSV is intentionally not processed by default.

In [ ]:
# Optional cleanup before another model run.
import gc

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model memory released.")